## 13. Feature Engineering:
This is the most critical step of the pipeline. Algorithms are only as good as the math we feed them. We extract structural, non-linear signals from the raw price.

**Our Core Features:**
1. `log_return`: Normalizes the exponential compounding of stock prices.
2. `vol_6h` & `vol_24h`: We use **two** rolling windows to capture both intra-day flash crashes (6h) and multi-day sustained panics (24h).
3. `vwap_deviation`: Institutional traders buy near the VWAP. If the price drifts far from the VWAP, it signals an unnatural imbalance.
4. **CRITICAL - Avoiding Look-Ahead Bias:** When calculating a 24-hour rolling volatility, the first 23 rows mathematically cannot be computed. Using `fillna(0)` to fix this causes a *fatal error*. If we pad with zeros, the ML model will see a sequence of $0.00$ volatility and instantly classify the start of our dataset as an extreme anomaly because it's 'unnaturally' stable. Instead, we use `dropna()` to strictly drop the initial warmup rows, forcing the model to only evaluate truly valid math.

In [1]:
# Notebook imports
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
# Notebook configuration
FEATURE_COLS = [
    "return_1h",
    "log_return",
    "vol_6h",
    "vol_24h",
    "spread_pct",
    "vwap_deviation",
    "high_low_range",
]
# Provide alias expected by other notebooks
FEATURES = FEATURE_COLS

In [3]:
_cwd = Path().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
DATA_DIR = PROJECT_ROOT / "data"
PLOTS_DIR = PROJECT_ROOT / "reports" / "figures"

# Self-load: ensure data is available

PROCESSED_CSV = DATA_DIR / "processed" / "msft_hourly(in)_processed.csv"
if "df" not in globals() or "quote_datetime" not in df.columns:
    if not PROCESSED_CSV.exists():
        raise FileNotFoundError(
            f"Processed CSV not found at {PROCESSED_CSV}. "
            "Run 1.0-abz-preprocessing.ipynb first to generate it."
        )
    df = pd.read_csv(PROCESSED_CSV, parse_dates=["quote_datetime"])
    print(f"Loaded processed data: {len(df)} rows")
else:
    if not pd.api.types.is_datetime64_any_dtype(df["quote_datetime"]):
        df["quote_datetime"] = pd.to_datetime(df["quote_datetime"])
    print(f"Using existing df: {len(df)} rows")

Loaded processed data: 15501 rows


In [4]:
def compute_features(data):
    features = data.copy()

    # Base transforms
    features["return_1h"] = features["close"].pct_change()
    features["log_return"] = np.log(features["close"] / features["close"].shift(1))
    features["vol_6h"] = features["log_return"].rolling(6).std()
    features["vol_24h"] = features["log_return"].rolling(24).std()

    # Quotes & Execution
    features["spread_pct"] = (features["ask"] - features["bid"]) / features["mid"] * 100
    features["vwap_deviation"] = (
        (features["close"] - features["vwap"]) / features["vwap"] * 100
    )
    features["high_low_range"] = (
        (features["high"] - features["low"]) / features["close"] * 100
    )

    return features


# Compute and strictly drop NaNs
df_feat = compute_features(df)
print(f"Rows before dropping NaN warmup windows: {len(df_feat)}")
df_model = df_feat.dropna(subset=FEATURE_COLS).copy().reset_index(drop=True)
print(f"Rows strictly valid for ML modeling: {len(df_model)}")
# Provide lowercase alias expected by ensemble notebook
feature_cols = FEATURE_COLS

Rows before dropping NaN warmup windows: 15501
Rows strictly valid for ML modeling: 15477


In [5]:
df_model.columns

Index(['underlying_symbol', 'quote_datetime', 'quote_date', 'quote_time',
       'open', 'high', 'low', 'close', 'trade_volume', 'vwap', 'bid', 'ask',
       'mid', 'return_1h', 'log_return', 'vol_6h', 'vol_24h', 'spread_pct',
       'vwap_deviation', 'high_low_range'],
      dtype='str')